# ApexPlanet Data Analytics Internship — Task 2
## SQL & Data Extraction

**Database:** `amazon_sales.db` (SQLite) — built from the Task 1 cleaned dataset
**Table:** `orders` (128,942 rows)

This notebook covers:
1. Database setup & connection (SQLAlchemy/sqlite3 via `db_utils.py`)
2. SQL fundamentals (SELECT, WHERE, JOIN, GROUP BY, subqueries/CTEs, window functions)
3. Advanced SQL for business questions
4. Views for frequently used queries
5. Query optimization (EXPLAIN QUERY PLAN + indexing)
6. Python + SQL integration — running queries from Jupyter into pandas DataFrames


In [1]:
import pandas as pd
import sqlite3
import sys
sys.path.insert(0, "../scripts")
from db_utils import get_connection, run_query, run_parameterized_query, execute_script

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

conn = get_connection()
print("Connected to database.")


Connected to database.


## 1. Database Setup & Data Import

In [2]:
# The orders table was created and populated from the Task 1 cleaned CSV.
# (See scripts/clean_data.py -> data/processed/amazon_sale_report_cleaned.csv,
#  loaded into SQLite via pandas.DataFrame.to_sql())
row_count = run_query("SELECT COUNT(*) AS row_count FROM orders", conn)
print(row_count.to_string(index=False))


 row_count
    128942


In [3]:
schema = run_query("PRAGMA table_info(orders)", conn)
print(schema[['name', 'type']].to_string(index=False))


               name    type
              index INTEGER
           order_id    TEXT
               date    TEXT
             status    TEXT
         fulfilment    TEXT
      sales_channel    TEXT
 ship_service_level    TEXT
              style    TEXT
                sku    TEXT
           category    TEXT
               size    TEXT
               asin    TEXT
     courier_status    TEXT
                qty INTEGER
           currency    TEXT
             amount    REAL
          ship_city    TEXT
         ship_state    TEXT
   ship_postal_code    REAL
       ship_country    TEXT
                b2b INTEGER
       fulfilled_by    TEXT
      has_promotion INTEGER
amount_outlier_flag INTEGER
        order_month    TEXT
         order_week    TEXT
          order_day    TEXT


In [4]:
# Build all views + indexes defined in scripts/queries.sql
execute_script("../scripts/queries.sql", conn)
print("Views and indexes created successfully from queries.sql")


Views and indexes created successfully from queries.sql


## 2. SQL Fundamentals

### 2.1 SELECT / WHERE / ORDER BY / LIMIT

In [5]:
sql = '''
SELECT order_id, date, category, amount, ship_state
FROM orders
WHERE status = 'Shipped - Delivered to Buyer'
ORDER BY amount DESC, date DESC
LIMIT 10;
'''
df = run_query(sql, conn)
print(df.to_string(index=False))


           order_id       date      category  amount     ship_state
403-4106553-1691525 2022-04-23           Set  5495.0         Punjab
408-3363121-6123562 2022-04-06 Western Dress  2860.0  Uttar Pradesh
403-2726196-9752350 2022-06-02           Set  2598.0    Maharashtra
405-5657207-4685151 2022-05-26           Set  2598.0    Maharashtra
402-0603541-4663511 2022-05-16           Set  2598.0    Maharashtra
406-0207843-8044301 2022-05-15           Set  2598.0      Rajasthan
406-3360895-1639546 2022-05-08           Set  2442.0    Maharashtra
408-0398992-2545935 2022-05-08           Set  2442.0 Andhra Pradesh
171-6011215-7144350 2022-05-20           Set  2372.0      Rajasthan
403-2895516-9774723 2022-05-18           Set  2372.0        Haryana


### 2.2 JOIN (self-join demo — dataset has no second table to join against)

In [6]:
sql = '''
SELECT a.order_id AS order_a, b.order_id AS order_b,
       a.sku, a.ship_state, a.amount AS amount_a, b.amount AS amount_b
FROM orders a
INNER JOIN orders b
    ON a.sku = b.sku
   AND a.ship_state = b.ship_state
   AND a.order_id < b.order_id
WHERE a.ship_state = 'DELHI'
LIMIT 10;
'''
df = run_query(sql, conn)
print(df.to_string(index=False))


Empty DataFrame
Columns: [order_a, order_b, sku, ship_state, amount_a, amount_b]
Index: []


### 2.3 GROUP BY / HAVING / aggregate functions

In [7]:
sql = '''
SELECT category,
       COUNT(*) AS cancelled_orders,
       ROUND(SUM(amount), 2) AS lost_revenue
FROM orders
WHERE status = 'Cancelled'
GROUP BY category
HAVING COUNT(*) > 500
ORDER BY cancelled_orders DESC;
'''
df = run_query(sql, conn)
print(df.to_string(index=False))


     category  cancelled_orders  lost_revenue
          Set              7336    3472451.03
        kurta              7249    1871646.70
Western Dress              2122    1006482.69
          Top              1276     443726.30


### 2.4 Subqueries and CTEs (WITH clause)

In [8]:
sql = '''
WITH state_revenue AS (
    SELECT ship_state, SUM(amount) AS revenue
    FROM orders
    WHERE amount > 0
    GROUP BY ship_state
),
ranked AS (
    SELECT ship_state, revenue,
           NTILE(4) OVER (ORDER BY revenue DESC) AS quartile
    FROM state_revenue
)
SELECT ship_state, ROUND(revenue, 2) AS revenue
FROM ranked
WHERE quartile = 1
ORDER BY revenue DESC;
'''
df = run_query(sql, conn)
print(df.to_string(index=False))


    ship_state     revenue
   Maharashtra 13335534.14
     Karnataka 10481114.37
     Telangana  6916615.65
 Uttar Pradesh  6816642.08
    Tamil Nadu  6515650.11
         Delhi  4393522.41
        Kerala  3830227.58
   West Bengal  3507880.44
Andhra Pradesh  3219831.72
       Haryana  2882092.99


### 2.5 Window functions (ROW_NUMBER, RANK, LAG)

In [9]:
sql = '''
WITH daily AS (
    SELECT order_day, SUM(amount) AS daily_revenue
    FROM orders
    WHERE amount > 0
    GROUP BY order_day
)
SELECT order_day,
       ROUND(daily_revenue, 2) AS daily_revenue,
       ROUND(LAG(daily_revenue) OVER (ORDER BY order_day), 2) AS prev_day_revenue,
       ROUND(daily_revenue - LAG(daily_revenue) OVER (ORDER BY order_day), 2) AS day_over_day_change
FROM daily
ORDER BY order_day
LIMIT 10;
'''
df = run_query(sql, conn)
print(df.to_string(index=False))


 order_day  daily_revenue  prev_day_revenue  day_over_day_change
2022-03-31      101683.85               NaN                  NaN
2022-04-01      865478.60         101683.85            763794.75
2022-04-02      913101.53         865478.60             47622.93
2022-04-03     1011763.38         913101.53             98661.85
2022-04-04      882059.17        1011763.38           -129704.21
2022-04-05      950544.05         882059.17             68484.88
2022-04-06      886985.26         950544.05            -63558.79
2022-04-07      909899.35         886985.26             22914.09
2022-04-08     1017857.61         909899.35            107958.26
2022-04-09      972076.48        1017857.61            -45781.13


## 3. Advanced SQL — Business Questions

> **Dataset limitation note:** this export has no customer-ID field (one row per order
> line, not per customer). Two queries below (`top customers`, `retention rate`) are
> adapted to the closest meaningful proxies available — shipping location revenue and
> SKU repeat-order rate — and are clearly labeled as such.


### 3.1 Monthly sales trend

In [10]:
sql = '''
SELECT order_month,
       COUNT(*) AS order_count,
       SUM(qty) AS units_sold,
       ROUND(SUM(amount), 2) AS total_revenue,
       ROUND(AVG(amount), 2) AS avg_order_value
FROM orders
WHERE amount > 0
GROUP BY order_month
ORDER BY order_month;
'''
df = run_query(sql, conn)
print(df.to_string(index=False))


order_month  order_count  units_sold  total_revenue  avg_order_value
    2022-03          162         156      101683.85           627.68
    2022-04        45222       43268    28831249.32           637.55
    2022-05        38777       37211    26219850.75           676.17
    2022-06        34645       33476    23421223.38           676.03


### 3.2 Top 10 revenue locations (proxy for 'top customers' — no customer ID in data)

In [11]:
sql = '''
SELECT ship_city, ship_state,
       COUNT(*) AS order_count,
       ROUND(SUM(amount), 2) AS total_revenue
FROM orders
WHERE amount > 0
GROUP BY ship_city, ship_state
ORDER BY total_revenue DESC
LIMIT 10;
'''
df = run_query(sql, conn)
print(df.to_string(index=False))


ship_city    ship_state  order_count  total_revenue
BENGALURU     Karnataka        10466     6849664.99
HYDERABAD     Telangana         7475     4941131.82
   MUMBAI   Maharashtra         5731     3704461.80
NEW DELHI         Delhi         5302     3613874.78
  CHENNAI    Tamil Nadu         5042     3098745.74
     PUNE   Maharashtra         3576     2338518.18
  KOLKATA   West Bengal         2201     1414978.87
 GURUGRAM       Haryana         1737     1221618.74
    THANE   Maharashtra         1563     1004355.29
  LUCKNOW Uttar Pradesh         1320      933926.34


### 3.3 'Retention' proxy — SKU repeat-order rate (no customer ID in data)

In [12]:
sql = '''
WITH sku_orders AS (
    SELECT sku, COUNT(*) AS order_count
    FROM orders
    GROUP BY sku
)
SELECT
    COUNT(*) AS total_skus,
    SUM(CASE WHEN order_count > 1 THEN 1 ELSE 0 END) AS repeat_skus,
    ROUND(100.0 * SUM(CASE WHEN order_count > 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS repeat_sku_rate_pct
FROM sku_orders;
'''
df = run_query(sql, conn)
print(df.to_string(index=False))


 total_skus  repeat_skus  repeat_sku_rate_pct
       7195         6302                87.59


### 3.4 Product category performance

In [13]:
sql = '''
SELECT category,
       COUNT(*) AS order_count,
       SUM(qty) AS units_sold,
       ROUND(SUM(amount), 2) AS total_revenue,
       ROUND(AVG(amount), 2) AS avg_order_value,
       ROUND(100.0 * SUM(CASE WHEN status = 'Cancelled' THEN 1 ELSE 0 END) / COUNT(*), 2) AS cancellation_rate_pct
FROM orders
GROUP BY category
ORDER BY total_revenue DESC;
'''
df = run_query(sql, conn)
print(df.to_string(index=False))


     category  order_count  units_sold  total_revenue  avg_order_value  cancellation_rate_pct
          Set        50272       45278    39195176.03           779.66                  14.59
        kurta        49859       45031    21291538.70           427.04                  14.54
Western Dress        15499       13942    11215337.69           723.62                  13.69
          Top        10620        9901     5346812.30           503.47                  12.02
 Ethnic Dress         1159        1053      791217.66           682.67                  12.51
       Blouse          926         863      458408.18           495.04                  12.53
       Bottom          440         398      150667.98           342.43                  13.64
        Saree          164         152      123933.76           755.69                  12.80
      Dupatta            3           3         915.00           305.00                   0.00


### 3.5 Moving averages and cumulative sums (7-day rolling revenue)

In [14]:
sql = '''
WITH daily AS (
    SELECT order_day, SUM(amount) AS daily_revenue
    FROM orders
    WHERE amount > 0
    GROUP BY order_day
)
SELECT order_day,
       ROUND(daily_revenue, 2) AS daily_revenue,
       ROUND(AVG(daily_revenue) OVER (
           ORDER BY order_day ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
       ), 2) AS rolling_7day_avg,
       ROUND(SUM(daily_revenue) OVER (
           ORDER BY order_day ROWS UNBOUNDED PRECEDING
       ), 2) AS cumulative_revenue
FROM daily
ORDER BY order_day
LIMIT 15;
'''
df = run_query(sql, conn)
print(df.to_string(index=False))


 order_day  daily_revenue  rolling_7day_avg  cumulative_revenue
2022-03-31      101683.85         101683.85           101683.85
2022-04-01      865478.60         483581.22           967162.45
2022-04-02      913101.53         626754.66          1880263.98
2022-04-03     1011763.38         723006.84          2892027.36
2022-04-04      882059.17         754817.31          3774086.53
2022-04-05      950544.05         787438.43          4724630.58
2022-04-06      886985.26         801659.41          5611615.84
2022-04-07      909899.35         917118.76          6521515.19
2022-04-08     1017857.61         938887.19          7539372.80
2022-04-09      972076.48         947312.19          8511449.28
2022-04-10     1075234.03         956379.42          9586683.31
2022-04-11      949559.27         966022.29         10536242.58
2022-04-12      885895.03         956786.72         11422137.61
2022-04-13      977017.30         969648.44         12399154.91
2022-04-14     1113487.56         998732

## 4. Views for Frequently Used Queries

In [15]:
print("View: monthly_category_sales")
print(run_query("SELECT * FROM monthly_category_sales ORDER BY order_month LIMIT 6;", conn).to_string(index=False))
print()
print("View: state_performance (top 5 by revenue)")
print(run_query("SELECT * FROM state_performance ORDER BY total_revenue DESC LIMIT 5;", conn).to_string(index=False))


View: monthly_category_sales
order_month      category  order_count  units_sold  total_revenue
    2022-03        Blouse            1           1         280.00
    2022-03  Ethnic Dress            1           1        1099.00
    2022-03           Set           68          68       53884.00
    2022-03           Top            9           9        4511.00
    2022-03 Western Dress            9           6        7653.28
    2022-03         kurta           74          71       34256.57

View: state_performance (top 5 by revenue)
   ship_state  order_count  total_revenue  avg_order_value  cancellation_rate_pct
  Maharashtra        22260    13335534.14           599.08                  13.32
    Karnataka        17326    10481114.37           604.94                  12.96
    Telangana        11330     6916615.65           610.47                  14.42
Uttar Pradesh        10638     6816642.08           640.78                  15.08
   Tamil Nadu        11483     6515650.11           567

## 5. Query Optimization — EXPLAIN QUERY PLAN & Indexing

In [16]:
plan = run_query('''
EXPLAIN QUERY PLAN
SELECT category, SUM(amount)
FROM orders
WHERE ship_state = 'MAHARASHTRA'
GROUP BY category;
''', conn)
print(plan.to_string(index=False))
print()
print("Indexes now present (from queries.sql):")
idx = run_query("SELECT name, tbl_name FROM sqlite_master WHERE type='index' AND tbl_name='orders';", conn)
print(idx.to_string(index=False))


 id  parent  notused                                                         detail
  7       0        0 SEARCH orders USING INDEX idx_orders_ship_state (ship_state=?)
 12       0        0                                   USE TEMP B-TREE FOR GROUP BY

Indexes now present (from queries.sql):
                  name tbl_name
 idx_orders_ship_state   orders
   idx_orders_category   orders
idx_orders_order_month   orders
        idx_orders_sku   orders
     idx_orders_status   orders


**Note on the query plan:** with `idx_orders_ship_state` in place (created in
`queries.sql`), SQLite's planner uses a `SEARCH` on that index instead of a full
table `SCAN`, which is significantly faster on a 128K-row table — this matters
increasingly as the dataset grows.

## 6. Python + SQL Integration

### 6.1 Parameterized queries (SQL-injection safe)

In [17]:
df = run_parameterized_query(
    "SELECT order_id, category, amount, ship_state FROM orders "
    "WHERE ship_state = :state AND amount > :min_amt ORDER BY amount DESC LIMIT 5",
    {"state": "KARNATAKA", "min_amt": 1000},
    conn,
)
print(df.to_string(index=False))


Empty DataFrame
Columns: [order_id, category, amount, ship_state]
Index: []


### 6.2 Reusable `db_utils.py` module

In [18]:
import inspect
from db_utils import get_connection, run_query, run_parameterized_query, execute_script
print("Functions available in db_utils.py:")
for fn in [get_connection, run_query, run_parameterized_query, execute_script]:
    print(f" - {fn.__name__}{inspect.signature(fn)}")


Functions available in db_utils.py:
 - get_connection(db_path: str = '/home/claude/apexplanet-data-analytics/notebooks/../scripts/../data/processed/amazon_sales.db')
 - run_query(sql: str, conn=None) -> pandas.DataFrame
 - run_parameterized_query(sql: str, params: dict, conn=None) -> pandas.DataFrame
 - execute_script(sql_path: str, conn=None)


## 7. Summary

- Built a local SQLite database (`amazon_sales.db`) from the Task 1 cleaned dataset
- Practiced SQL fundamentals: SELECT/WHERE/ORDER BY, JOIN, GROUP BY/HAVING, CTEs, and
  window functions (ROW_NUMBER, RANK, LAG, NTILE)
- Answered business questions: monthly trend, top revenue locations, category
  performance, cancellation rates, and rolling/cumulative revenue
- Created 3 reusable views (`monthly_category_sales`, `state_performance`,
  `daily_sales_summary`) and 5 indexes for query optimization
- Verified indexing improves the query plan via `EXPLAIN QUERY PLAN`
- Built a reusable `db_utils.py` module for safe, parameterized Python↔SQL access

**Next:** Task 3 will turn these queries into an interactive Power BI / Tableau dashboard.
